# Comprehensive Daytime Heat Threshold Analysis: Multi-Level Temperature Response

## Scientific Background

Daytime heat exposure (8am-8pm) represents the **primary thermal stress period** for cattle. This analysis examines mortality responses across **multiple temperature thresholds** to identify:

### Temperature Thresholds and Physiological Significance

**Heat Stress Thresholds** (8am-8pm, 12 hours):
1. **>25°C (77°F)**: Mild stress threshold
   - Dairy cattle begin experiencing thermal discomfort
   - Increased respiration rate
   - Reduced feed intake begins

2. **>30°C (86°F)**: Moderate stress threshold
   - Both dairy and beef cattle affected
   - Panting behavior increases
   - Significant reduction in milk production
   - Behavioral changes (shade-seeking)

3. **>35°C (95°F)**: Severe stress threshold
   - Critical thermal stress
   - Body temperature regulation challenged
   - Drooling, open-mouth breathing
   - Risk of heat exhaustion

4. **>40°C (104°F)**: Extreme/lethal threshold
   - Life-threatening conditions
   - Heat stroke risk
   - Mortality can occur within hours
   - Emergency intervention required

**Cold Stress Thresholds**:
1. **<5°C (41°F)**: Mild cold exposure
   - Increased metabolic heat production
   - Higher feed requirements

2. **<0°C (32°F)**: Moderate cold stress
   - Significant energy expenditure for thermoregulation
   - Risk for young/thin animals

3. **<-5°C (23°F)**: Severe cold stress
   - Risk of frostbite
   - Hypothermia risk without shelter

### Research Questions

1. How does mortality change across incremental temperature thresholds?
2. Are there critical inflection points where mortality accelerates?
3. Do heat and cold stress show symmetric or asymmetric mortality effects?
4. How do threshold effects vary by season and region?
5. Can we identify optimal temperature ranges for minimizing mortality?

### Hypotheses

**H1**: Mortality increases non-linearly with temperature threshold severity  
**H2**: Extreme heat (>35°C) has disproportionate mortality impact  
**H3**: Cold stress mortality is lower than heat stress mortality in these regions  
**H4**: Multiple consecutive days above threshold compound mortality risk  
**H5**: Regional differences exist in threshold sensitivities (humid vs arid)

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr, kruskal, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Import project configuration
import sys
sys.path.append('../../')
from config import (
    STATE_NAMES, STATE_ABBRS, STATE_REGIONS, 
    FOCUS_STATES, CATTLE_REGIONS, CUSTOM_REGIONS, SEASONS
)

# Set plotting style
sns.set_style('whitegrid')
sns.set_palette('RdYlBu_r')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully")
print(f"Focus states (n={len(FOCUS_STATES)}): {sorted([STATE_ABBRS[s] for s in FOCUS_STATES.values()])}")

## 1. Data Loading and Integration

In [ ]:
# Load merged cattle-heat dataset
cattle_heat = pd.read_csv('../../cattle_data/cattle_heat_merged.csv', parse_dates=['week_ending'])
print(f"Loaded {len(cattle_heat)} weeks of data ({cattle_heat['week_ending'].min()} to {cattle_heat['week_ending'].max()})")

# Check available daytime heat columns
print(f"\nAvailable daytime heat columns:")
daytime_cols = [col for col in cattle_heat.columns if 'daytime' in col.lower()]
for col in daytime_cols:
    print(f"  - {col}")

# Add temporal features
cattle_heat['year'] = cattle_heat['week_ending'].dt.year
cattle_heat['month'] = cattle_heat['week_ending'].dt.month
cattle_heat['week_of_year'] = cattle_heat['week_ending'].dt.isocalendar().week

def get_season(month):
    if month in SEASONS['Winter']:
        return 'Winter'
    elif month in SEASONS['Spring']:
        return 'Spring'
    elif month in SEASONS['Summer']:
        return 'Summer'
    else:
        return 'Fall'

cattle_heat['season'] = cattle_heat['month'].apply(get_season)
cattle_heat['decade'] = (cattle_heat['year'] // 10) * 10

# Focus on Regions 4 and 6
cattle_focus = cattle_heat[cattle_heat['region'].isin([4, 6])].copy()

print(f"\nFocused on Regions 4 & 6: {len(cattle_focus)} region-weeks")
print(f"Date range: {cattle_focus['year'].min()}-{cattle_focus['year'].max()}")

## 2. Heat Threshold Correlation Analysis

Examine correlations between each temperature threshold and cattle mortality.

In [ ]:
# Define all temperature threshold variables
heat_thresholds = [
    ('mean_daytime_hours_above_25', '>25°C', 'Mild Heat'),
    ('mean_daytime_hours_above_30', '>30°C', 'Moderate Heat'),
    ('mean_daytime_hours_above_35', '>35°C', 'Severe Heat'),
    ('mean_daytime_hours_above_40', '>40°C', 'Extreme Heat')
]

cold_thresholds = [
    ('mean_daytime_hours_below_5', '<5°C', 'Mild Cold'),
    ('mean_daytime_hours_below_0', '<0°C', 'Moderate Cold'),
    ('mean_daytime_hours_below_neg5', '<-5°C', 'Severe Cold')
]

mortality_var = 'total_beef_dairy'

# Calculate correlations for heat thresholds
print("="*80)
print("DAYTIME HEAT THRESHOLD CORRELATIONS WITH MORTALITY")
print("="*80)

heat_correlations = []

for var, label, category in heat_thresholds:
    if var in cattle_focus.columns:
        valid_data = cattle_focus[[var, mortality_var]].dropna()
        if len(valid_data) > 10:
            pearson_r, pearson_p = pearsonr(valid_data[var], valid_data[mortality_var])
            spearman_r, spearman_p = spearmanr(valid_data[var], valid_data[mortality_var])
            
            heat_correlations.append({
                'Threshold': label,
                'Category': category,
                'Variable': var,
                'Pearson_r': pearson_r,
                'Pearson_p': pearson_p,
                'Spearman_r': spearman_r,
                'Spearman_p': spearman_p,
                'N': len(valid_data),
                'Mean_Hours': valid_data[var].mean(),
                'Max_Hours': valid_data[var].max()
            })
            
            print(f"\n{category} ({label}):")
            print(f"  Pearson r = {pearson_r:.4f}, p = {pearson_p:.4e}")
            print(f"  Spearman r = {spearman_r:.4f}, p = {spearman_p:.4e}")
            print(f"  Mean hours/week: {valid_data[var].mean():.2f}")
            print(f"  Max hours/week: {valid_data[var].max():.2f}")
    else:
        print(f"\n{category} ({label}): Variable '{var}' not found in dataset")

heat_corr_df = pd.DataFrame(heat_correlations)

# Calculate correlations for cold thresholds
print("\n" + "="*80)
print("DAYTIME COLD THRESHOLD CORRELATIONS WITH MORTALITY")
print("="*80)

cold_correlations = []

for var, label, category in cold_thresholds:
    if var in cattle_focus.columns:
        valid_data = cattle_focus[[var, mortality_var]].dropna()
        if len(valid_data) > 10:
            pearson_r, pearson_p = pearsonr(valid_data[var], valid_data[mortality_var])
            spearman_r, spearman_p = spearmanr(valid_data[var], valid_data[mortality_var])
            
            cold_correlations.append({
                'Threshold': label,
                'Category': category,
                'Variable': var,
                'Pearson_r': pearson_r,
                'Pearson_p': pearson_p,
                'Spearman_r': spearman_r,
                'Spearman_p': spearman_p,
                'N': len(valid_data),
                'Mean_Hours': valid_data[var].mean(),
                'Max_Hours': valid_data[var].max()
            })
            
            print(f"\n{category} ({label}):")
            print(f"  Pearson r = {pearson_r:.4f}, p = {pearson_p:.4e}")
            print(f"  Spearman r = {spearman_r:.4f}, p = {spearman_p:.4e}")
            print(f"  Mean hours/week: {valid_data[var].mean():.2f}")
            print(f"  Max hours/week: {valid_data[var].max():.2f}")
    else:
        print(f"\n{category} ({label}): Variable '{var}' not found in dataset")

cold_corr_df = pd.DataFrame(cold_correlations)

In [ ]:
# Visualize threshold correlations
import os
os.makedirs('../../figures/daytime_heat_comprehensive', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Heat threshold correlations
ax = axes[0]
if len(heat_corr_df) > 0:
    x = np.arange(len(heat_corr_df))
    width = 0.35
    
    ax.bar(x - width/2, heat_corr_df['Pearson_r'], width, label='Pearson', color='coral', alpha=0.8)
    ax.bar(x + width/2, heat_corr_df['Spearman_r'], width, label='Spearman', color='steelblue', alpha=0.8)
    
    ax.set_xlabel('Heat Threshold', fontsize=11, fontweight='bold')
    ax.set_ylabel('Correlation with Mortality', fontsize=11, fontweight='bold')
    ax.set_title('Daytime Heat Thresholds: Correlation Strength', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(heat_corr_df['Threshold'], fontsize=10)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.legend(loc='upper left')
    ax.grid(axis='y', alpha=0.3)

# Plot 2: Cold threshold correlations
ax = axes[1]
if len(cold_corr_df) > 0:
    x = np.arange(len(cold_corr_df))
    
    ax.bar(x - width/2, cold_corr_df['Pearson_r'], width, label='Pearson', color='skyblue', alpha=0.8)
    ax.bar(x + width/2, cold_corr_df['Spearman_r'], width, label='Spearman', color='navy', alpha=0.8)
    
    ax.set_xlabel('Cold Threshold', fontsize=11, fontweight='bold')
    ax.set_ylabel('Correlation with Mortality', fontsize=11, fontweight='bold')
    ax.set_title('Daytime Cold Thresholds: Correlation Strength', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(cold_corr_df['Threshold'], fontsize=10)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/daytime_heat_comprehensive/01_threshold_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 01_threshold_correlations.png")

## 3. Dose-Response Curves

Examine how mortality changes with increasing exposure hours at each threshold.

In [ ]:
# Create dose-response curves for heat thresholds
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (var, label, category) in enumerate(heat_thresholds):
    if var not in cattle_focus.columns:
        continue
    
    ax = axes[idx]
    
    # Scatter plot
    valid_data = cattle_focus[[var, mortality_var]].dropna()
    if len(valid_data) > 10:
        ax.scatter(valid_data[var], valid_data[mortality_var], 
                  alpha=0.3, s=15, color='coral')
        
        # Add polynomial fit
        if valid_data[var].max() > 0:
            z = np.polyfit(valid_data[var], valid_data[mortality_var], 2)
            p = np.poly1d(z)
            x_line = np.linspace(0, valid_data[var].max(), 100)
            ax.plot(x_line, p(x_line), 'r--', linewidth=2, label='Quadratic Fit')
        
        # Add bin aggregates
        bins = pd.cut(valid_data[var], bins=10)
        binned_mean = valid_data.groupby(bins)[mortality_var].mean()
        bin_centers = [(interval.left + interval.right) / 2 for interval in binned_mean.index]
        ax.plot(bin_centers, binned_mean.values, 'bo-', linewidth=2, markersize=8, 
               label='Binned Mean', alpha=0.7)
        
        # Correlation
        r, p_val = pearsonr(valid_data[var], valid_data[mortality_var])
        ax.text(0.05, 0.95, f'r = {r:.3f}\np = {p_val:.4f}', 
               transform=ax.transAxes, fontsize=10, va='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_xlabel(f'Weekly Hours {label}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Cattle Mortality (1000 head)', fontsize=11, fontweight='bold')
    ax.set_title(f'{category} Dose-Response', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/daytime_heat_comprehensive/02_heat_dose_response_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 02_heat_dose_response_curves.png")

## 4. Summary and Key Findings

In [ ]:
print("="*80)
print("KEY FINDINGS: COMPREHENSIVE DAYTIME HEAT THRESHOLD ANALYSIS")
print("="*80)

print("\n1. HEAT THRESHOLD CORRELATIONS:")
if len(heat_corr_df) > 0:
    for _, row in heat_corr_df.iterrows():
        print(f"   {row['Category']} ({row['Threshold']}):")
        print(f"     Pearson r = {row['Pearson_r']:.4f}")
        print(f"     Mean exposure: {row['Mean_Hours']:.1f} hours/week")

print("\n2. COLD THRESHOLD CORRELATIONS:")
if len(cold_corr_df) > 0:
    for _, row in cold_corr_df.iterrows():
        print(f"   {row['Category']} ({row['Threshold']}):")
        print(f"     Pearson r = {row['Pearson_r']:.4f}")
        print(f"     Mean exposure: {row['Mean_Hours']:.1f} hours/week")

print("\n3. STRONGEST PREDICTOR:")
all_corr = pd.concat([heat_corr_df, cold_corr_df], ignore_index=True)
if len(all_corr) > 0:
    strongest = all_corr.loc[all_corr['Pearson_r'].abs().idxmax()]
    print(f"   {strongest['Category']} ({strongest['Threshold']})")
    print(f"   Correlation: r = {strongest['Pearson_r']:.4f}")

print("\n" + "="*80)
print("✓ Analysis complete! All figures saved to figures/daytime_heat_comprehensive/")

## 5. Export Results

In [ ]:
# Save correlation results
all_threshold_corr = pd.concat([heat_corr_df, cold_corr_df], ignore_index=True)
all_threshold_corr.to_csv('../../cattle_data/daytime_threshold_correlations.csv', index=False)
print(f"Threshold correlations saved to: cattle_data/daytime_threshold_correlations.csv")

print("\n✓ Analysis complete!")